<a href="https://colab.research.google.com/github/VaggelisApostolou/Challenges-in-Detecting-Toxic-Language-in-Greek-Sports-Social-Media/blob/main/Agreements.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Imports**

In [ ]:
!pip install --upgrade transformers datasets accelerate

In [ ]:
import os, random, numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
import math
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
import torch.nn as nn
import torch.nn.functional as F
from transformers import (
    XLMRobertaTokenizerFast,
    XLMRobertaConfig,
    XLMRobertaForSequenceClassification,
    XLMRobertaForMaskedLM,
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoModelForMaskedLM,
    get_linear_schedule_with_warmup,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)
from datasets import load_dataset
import glob
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_recall_curve, classification_report, precision_score, recall_score, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight

def set_seed(seed: int = 42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
XLM_CHECKPOINT = "xlm-roberta-base"
GREEKB_CHECKPOINT = "nlpaueb/bert-base-greek-uncased-v1"
MMB_CHECKPOINT = "jhu-clsp/mmBERT-base"
MASKED_XLM = "/content/drive/MyDrive/Toxic language in football Research/models/my_domain_adapted_model"
MASKED_GREEKB = "/content/drive/MyDrive/Toxic language in football Research/models/Greek-BERT"
MASKED_MMB = "/content/drive/MyDrive/Toxic language in football Research/models/mmBERT"

In [ ]:
DATA_PATH = "/content/drive/MyDrive/Toxic language in football Research/final_labeled_batches/aggrements_final.csv"

# **Dataset Analysis**

In [ ]:
df = pd.read_csv(DATA_PATH)

In [ ]:
print(df.value_counts("label"))

label
Non-toxic    5261
Toxic         288
Name: count, dtype: int64


# **Model**

## *Configuration*

In [ ]:
SEED      = 42
EPOCHS    = 8
BATCH_SZ  = 32
LR        = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
MAX_LEN   = 256
PATIENCE  = 3
GRAD_CLIP = 1.0

LOSS_TYPE = "focal"   # "ce" or "focal"
FOCAL_GAMMA = 2.0

# Toxic class emphasis
TOXIC_WEIGHT_MULT = 1.5
USE_SAMPLER = True

# Threshold objective
FBETA = 2.0              # tune threshold on F_beta (beta=2 -> recall emphasis)

set_seed(SEED)
print("Device:", device)
print("Loss:", LOSS_TYPE, "| gamma:", FOCAL_GAMMA, "| toxic_w_mult:", TOXIC_WEIGHT_MULT, "| Fbeta:", FBETA)


Device: cuda
Loss: focal | gamma: 2.0 | toxic_w_mult: 1.5 | Fbeta: 2.0


## *Load Datasets*

In [ ]:
def load_dataset(path: str):
    df = pd.read_csv(path)
    if not {"text", "label"}.issubset(df.columns):
        if df.shape[1] == 2:
            df.columns = ["text", "label"]
        else:
            raise ValueError(f"Το CSV πρέπει να έχει στήλες 'text' και 'label'. Βρέθηκαν: {df.columns.tolist()}")

    df = df.dropna(subset=["label", "text"]).copy()

    label2id = {"Non-toxic": 0, "Toxic": 1}

    if not set(df["label"].unique()).issubset(set(label2id.keys())):
        raise ValueError(f"Τα labels πρέπει να είναι {list(label2id.keys())}, βρέθηκαν: {df['label'].unique()}")

    df["label_id"] = df["label"].map(label2id)

    return df, label2id

In [ ]:
df, label2id = load_dataset(DATA_PATH)
print("Total shape:", df.shape)
print("Total label distribution:\n", df["label"].value_counts(normalize=True).round(3))
df.head()

Total shape: (5549, 4)
Total label distribution:
 label
Non-toxic    0.948
Toxic        0.052
Name: proportion, dtype: float64


,id,text,label,label_id
0,7438,@user Come on Captain... 💪💪💪🇬🇲🇬🇲🇬🇲🇬🇲🇬🇲,Non-toxic,0
1,7439,@user Έλα πίσω παικταρα,Non-toxic,0
2,7440,@user (L)!!!,Non-toxic,0
3,7443,@user tlogela Corona for 2mins and come do you...,Toxic,1
4,7445,@user Get well soon 🤝🖤,Non-toxic,0


## *Train-Test Split*

In [ ]:
# First: train (70%) vs temp (30%)
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df["label_id"], random_state=SEED
)
# Then: temp -> val (15%) and test (15%) i.e., split temp 50/50 stratified
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label_id"], random_state=SEED
)

def pct(x): return (x.value_counts(normalize=True).rename({0:"non-toxic",1:"toxic"})*100).round(1)

print("Split sizes:", len(train_df), len(val_df), len(test_df))
print("Train dist (%):\n", pct(train_df["label_id"]))
print("Val   dist (%):\n", pct(val_df["label_id"]))
print("Test  dist (%):\n", pct(test_df["label_id"]))

Split sizes: 3884 832 833
Train dist (%):
 label_id
non-toxic    94.8
toxic         5.2
Name: proportion, dtype: float64
Val   dist (%):
 label_id
non-toxic    94.8
toxic         5.2
Name: proportion, dtype: float64
Test  dist (%):
 label_id
non-toxic    94.8
toxic         5.2
Name: proportion, dtype: float64


## *Class Weights*

In [ ]:
class_weights = compute_class_weight(class_weight="balanced", classes=np.array([0,1]), y=train_df["label_id"].values)
class_weights = torch.tensor(class_weights, dtype=torch.float)
class_weights[1] = class_weights[1] * TOXIC_WEIGHT_MULT  # boost toxic
class_weights = class_weights.to(device)
print("Class weights (train) with boost:", class_weights.tolist())

Class weights (train) with boost: [0.5274307727813721, 14.420791625976562]


## *Focal Loss*

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction="mean"):
        super().__init__()
        self.alpha = alpha; self.gamma = gamma; self.reduction = reduction
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.alpha, reduction="none")
        pt = torch.exp(-ce)
        loss = (1 - pt) ** self.gamma * ce
        if self.reduction == "mean": return loss.mean()
        if self.reduction == "sum": return loss.sum()
        return loss

if LOSS_TYPE.lower() == "focal":
    loss_fn = FocalLoss(alpha=class_weights, gamma=FOCAL_GAMMA)
else:
    loss_fn = lambda logits, targets: F.cross_entropy(logits, targets, weight=class_weights)
loss_fn

FocalLoss()

# **Model 1 (XLM-RoBERTa)**

## **Classic**

In [ ]:
MODEL_NAME = XLM_CHECKPOINT

### *Model Setup*

In [ ]:
tokenizer = XLMRobertaTokenizerFast.from_pretrained(MODEL_NAME)
config = XLMRobertaConfig.from_pretrained(
    MODEL_NAME, num_labels=2, label2id=label2id, id2label={v:k for k,v in label2id.items()}
)
model = XLMRobertaForSequenceClassification.from_pretrained(MODEL_NAME, config=config).to(device)

class CommentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=160):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        text = str(self.texts[idx]); label = int(self.labels[idx])
        enc = self.tokenizer(text, truncation=True, padding='max_length', max_length=self.max_length, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'labels': torch.tensor(label, dtype=torch.long)}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### *Dataset Split*

In [ ]:
train_ds = CommentDataset(train_df["text"], train_df["label_id"], tokenizer, MAX_LEN)
val_ds   = CommentDataset(val_df["text"],   val_df["label_id"],   tokenizer, MAX_LEN)
test_ds  = CommentDataset(test_df["text"],  test_df["label_id"],  tokenizer, MAX_LEN)

# Sampler (train only)
if USE_SAMPLER:
    y_train = train_df["label_id"].values
    class_sample_count = np.bincount(y_train)     # e.g., [#non, #tox]
    weights = 1.0 / class_sample_count
    sample_weights = weights[y_train]
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SZ, sampler=sampler)
else:
    train_loader = DataLoader(train_ds, batch_size=BATCH_SZ, shuffle=True)

val_loader   = DataLoader(val_ds,   batch_size=BATCH_SZ)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SZ)

### *Evaluation Setup*

In [ ]:
@torch.no_grad()
def evaluate(model, dataloader, device, report=False, threshold=None):
    model.eval()
    all_probs, all_preds, all_true = [], [], []

    running_loss = 0.0
    for batch in dataloader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits  = outputs.logits
        loss    = F.cross_entropy(logits, labels)
        running_loss += loss.item()

        probs = torch.softmax(logits, dim=-1)[:, 1]
        all_probs.extend(probs.detach().cpu().numpy())

        preds = torch.argmax(logits, dim=1) if threshold is None else (probs >= threshold).long()
        all_preds.extend(preds.detach().cpu().numpy())
        all_true.extend(labels.detach().cpu().numpy())

    avg_loss = running_loss / max(1, len(dataloader))

    acc       = accuracy_score(all_true, all_preds)
    f1_macro  = f1_score(all_true, all_preds, average="macro",  zero_division=0)
    f1_toxic  = f1_score(all_true, all_preds, pos_label=1,      zero_division=0)
    prec_tox  = precision_score(all_true, all_preds, pos_label=1, zero_division=0)
    rec_tox   = recall_score(all_true, all_preds, pos_label=1,    zero_division=0)

    if report:
        print(classification_report(all_true, all_preds, digits=2))

    return {
        "loss": avg_loss,
        "acc": acc,
        "f1_macro": f1_macro,
        "f1_toxic": f1_toxic,
        "precision_toxic": prec_tox,
        "recall_toxic": rec_tox,
        "y_true": np.array(all_true),
        "y_probs": np.array(all_probs),
    }

def best_threshold_fbeta(y_true, p_toxic, beta=2.0):
    prec, rec, thr = precision_recall_curve(y_true, p_toxic)
    fbeta = (1+beta**2) * (prec*rec) / (beta**2 * prec + rec + 1e-9)
    if len(thr) == 0: return 0.5
    return float(thr[int(np.nanargmax(fbeta))])


### *Training*

In [ ]:
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
num_warmup = int(WARMUP_RATIO * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=num_warmup, num_training_steps=total_steps)

best_metric = -1.0
best_state = None
patience_ctr = 0

for epoch in range(1, EPOCHS+1):
    model.train(); running_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad(set_to_none=True)
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        out = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(out.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step(); scheduler.step()
        running_loss += loss.item()
    train_loss = running_loss / max(1, len(train_loader))

    val_metrics = evaluate(model, val_loader, device, report=False, threshold=None)
    print(
    f"Epoch {epoch:>2} | "
    f"TrainLoss {train_loss:.4f} | "
    f"ValAcc {val_metrics['acc']:.4f} | "
    f"ValF1_macro {val_metrics['f1_macro']:.4f} | "
    f"ValF1_toxic {val_metrics['f1_toxic']:.4f} | "
    f"ValPrec_toxic {val_metrics['precision_toxic']:.4f} | "
    f"ValRec_toxic {val_metrics['recall_toxic']:.4f}"
    )
    try:
      val_auroc = roc_auc_score(val_metrics["y_true"], val_metrics["y_probs"])
      print(f"Val AUROC: {val_auroc:.4f}")
    except ValueError:
      print("Val AUROC: not defined (only one class present in y_true)")

    current = val_metrics["f1_macro"]
    if current > best_metric:
        best_metric = current; patience_ctr = 0
        best_state = {k: v.cpu() for k,v in model.state_dict().items()}
    else:
        patience_ctr += 1
        if patience_ctr > PATIENCE:
            print(f"Early stopping at epoch {epoch}. Best Val macro-F1: {best_metric:.4f}")
            break

if best_state is not None:
    model.load_state_dict(best_state); model.to(device)
    print("Loaded best checkpoint (Val macro-F1=%.4f)" % best_metric)

Epoch  1 | TrainLoss 2.2414 | ValAcc 0.0517 | ValF1_macro 0.0491 | ValF1_toxic 0.0983 | ValPrec_toxic 0.0517 | ValRec_toxic 1.0000
Val AUROC: 0.9410
Epoch  2 | TrainLoss 0.2633 | ValAcc 0.8365 | ValF1_macro 0.6382 | ValF1_toxic 0.3704 | ValPrec_toxic 0.2312 | ValRec_toxic 0.9302
Val AUROC: 0.9447
Epoch  3 | TrainLoss 0.1844 | ValAcc 0.9075 | ValF1_macro 0.7127 | ValF1_toxic 0.4762 | ValPrec_toxic 0.3365 | ValRec_toxic 0.8140
Val AUROC: 0.9454
Epoch  4 | TrainLoss 0.1031 | ValAcc 0.9219 | ValF1_macro 0.7449 | ValF1_toxic 0.5324 | ValPrec_toxic 0.3854 | ValRec_toxic 0.8605
Val AUROC: 0.9499
Epoch  5 | TrainLoss 0.0765 | ValAcc 0.9579 | ValF1_macro 0.8121 | ValF1_toxic 0.6465 | ValPrec_toxic 0.5714 | ValRec_toxic 0.7442
Val AUROC: 0.9538
Epoch  6 | TrainLoss 0.0833 | ValAcc 0.9579 | ValF1_macro 0.8007 | ValF1_toxic 0.6237 | ValPrec_toxic 0.5800 | ValRec_toxic 0.6744
Val AUROC: 0.9478
Epoch  7 | TrainLoss 0.0522 | ValAcc 0.9567 | ValF1_macro 0.8010 | ValF1_toxic 0.6250 | ValPrec_toxic 0.56

### *Validation and Testing*

In [ ]:
# Tune threshold on VALIDATION set (Fbeta)
val_eval  = evaluate(model, val_loader, device, report=False, threshold=None)
best_thr  = best_threshold_fbeta(val_eval["y_true"], val_eval["y_probs"], beta=FBETA)
val_tuned = evaluate(model, val_loader, device, report=True, threshold=best_thr)

print("\n=== Validation (for tuning on Fbeta) ===")
print(f"Default thr=0.5 -> Val Acc: {val_eval['acc']:.4f} | Val Macro-F1: {val_eval['f1_macro']:.4f} | Val F1-toxic: {val_eval['f1_toxic']:.4f}")
print(f"Best thr={best_thr:.3f} (beta={FBETA}) -> Val Acc: {val_tuned['acc']:.4f} | Val Macro-F1: {val_tuned['f1_macro']:.4f} | Val F1-toxic: {val_tuned['f1_toxic']:.4f}")
try:
    val_auroc = roc_auc_score(val_eval["y_true"], val_eval["y_probs"])
    print(f"Val AUROC: {val_auroc:.4f}")
except ValueError:
    print("Val AUROC: not defined (only one class present)")

# FINAL TEST (use the tuned threshold)
test_eval_default = evaluate(model, test_loader, device, report=True, threshold=None)
test_eval_tuned   = evaluate(model, test_loader, device, report=True, threshold=best_thr)

print("\n=== FINAL TEST (unseen) ===")
print(f"Default thr=0.5 -> Test Acc: {test_eval_default['acc']:.4f} | Test Macro-F1: {test_eval_default['f1_macro']:.4f} | Test F1-toxic: {test_eval_default['f1_toxic']:.4f}")
print(f"Tuned thr={best_thr:.3f} (beta={FBETA}) -> Test Acc: {test_eval_tuned['acc']:.4f} | Test Macro-F1: {test_eval_tuned['f1_macro']:.4f} | Test F1-toxic: {test_eval_tuned['f1_toxic']:.4f}")
try:
    test_auroc = roc_auc_score(test_eval_default["y_true"], test_eval_default["y_probs"])
    print(f"Test AUROC: {test_auroc:.4f}")
except ValueError:
    print("Test AUROC: not defined (only one class present)")

              precision    recall  f1-score   support

           0       0.99      0.95      0.97       789
           1       0.50      0.84      0.63        43

    accuracy                           0.95       832
   macro avg       0.75      0.90      0.80       832
weighted avg       0.97      0.95      0.95       832


=== Validation (for tuning on Fbeta) ===
Default thr=0.5 -> Val Acc: 0.9579 | Val Macro-F1: 0.8121 | Val F1-toxic: 0.6465
Best thr=0.186 (beta=2.0) -> Val Acc: 0.9483 | Val Macro-F1: 0.7992 | Val F1-toxic: 0.6261
Val AUROC: 0.9538
              precision    recall  f1-score   support

           0       0.97      0.98      0.97       790
           1       0.53      0.47      0.49        43

    accuracy                           0.95       833
   macro avg       0.75      0.72      0.73       833
weighted avg       0.95      0.95      0.95       833

              precision    recall  f1-score   support

           0       0.98      0.96      0.97       790
     

## **Masked**

In [ ]:
MODEL_NAME = MASKED_XLM

### *Model Setup*

In [ ]:
tokenizer = XLMRobertaTokenizerFast.from_pretrained(MODEL_NAME)
config = XLMRobertaConfig.from_pretrained(
    MODEL_NAME, num_labels=2, label2id=label2id, id2label={v:k for k,v in label2id.items()}
)
model = XLMRobertaForSequenceClassification.from_pretrained(MODEL_NAME, config=config).to(device)

class CommentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=160):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        text = str(self.texts[idx]); label = int(self.labels[idx])
        enc = self.tokenizer(text, truncation=True, padding='max_length', max_length=self.max_length, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'labels': torch.tensor(label, dtype=torch.long)}


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: /content/drive/MyDrive/Toxic language in football Research/models/my_domain_adapted_model
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### *Dataset Split*

In [ ]:
train_ds = CommentDataset(train_df["text"], train_df["label_id"], tokenizer, MAX_LEN)
val_ds   = CommentDataset(val_df["text"],   val_df["label_id"],   tokenizer, MAX_LEN)
test_ds  = CommentDataset(test_df["text"],  test_df["label_id"],  tokenizer, MAX_LEN)

# Sampler (train only)
if USE_SAMPLER:
    y_train = train_df["label_id"].values
    class_sample_count = np.bincount(y_train)     # e.g., [#non, #tox]
    weights = 1.0 / class_sample_count
    sample_weights = weights[y_train]
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SZ, sampler=sampler)
else:
    train_loader = DataLoader(train_ds, batch_size=BATCH_SZ, shuffle=True)

val_loader   = DataLoader(val_ds,   batch_size=BATCH_SZ)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SZ)

### *Evaluation Setup*

In [ ]:
@torch.no_grad()
def evaluate(model, dataloader, device, report=False, threshold=None):
    model.eval()
    all_probs, all_preds, all_true = [], [], []

    running_loss = 0.0
    for batch in dataloader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits  = outputs.logits
        loss    = F.cross_entropy(logits, labels)
        running_loss += loss.item()

        probs = torch.softmax(logits, dim=-1)[:, 1]
        all_probs.extend(probs.detach().cpu().numpy())

        preds = torch.argmax(logits, dim=1) if threshold is None else (probs >= threshold).long()
        all_preds.extend(preds.detach().cpu().numpy())
        all_true.extend(labels.detach().cpu().numpy())

    avg_loss = running_loss / max(1, len(dataloader))

    acc       = accuracy_score(all_true, all_preds)
    f1_macro  = f1_score(all_true, all_preds, average="macro",  zero_division=0)
    f1_toxic  = f1_score(all_true, all_preds, pos_label=1,      zero_division=0)
    prec_tox  = precision_score(all_true, all_preds, pos_label=1, zero_division=0)
    rec_tox   = recall_score(all_true, all_preds, pos_label=1,    zero_division=0)

    if report:
        print(classification_report(all_true, all_preds, digits=2))

    return {
        "loss": avg_loss,
        "acc": acc,
        "f1_macro": f1_macro,
        "f1_toxic": f1_toxic,
        "precision_toxic": prec_tox,
        "recall_toxic": rec_tox,
        "y_true": np.array(all_true),
        "y_probs": np.array(all_probs),
    }

def best_threshold_fbeta(y_true, p_toxic, beta=2.0):
    prec, rec, thr = precision_recall_curve(y_true, p_toxic)
    fbeta = (1+beta**2) * (prec*rec) / (beta**2 * prec + rec + 1e-9)
    if len(thr) == 0: return 0.5
    return float(thr[int(np.nanargmax(fbeta))])


### *Training*

In [ ]:
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
num_warmup = int(WARMUP_RATIO * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=num_warmup, num_training_steps=total_steps)

best_metric = -1.0
best_state = None
patience_ctr = 0

for epoch in range(1, EPOCHS+1):
    model.train(); running_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad(set_to_none=True)
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        out = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(out.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step(); scheduler.step()
        running_loss += loss.item()
    train_loss = running_loss / max(1, len(train_loader))

    val_metrics = evaluate(model, val_loader, device, report=False, threshold=None)
    print(
    f"Epoch {epoch:>2} | "
    f"TrainLoss {train_loss:.4f} | "
    f"ValAcc {val_metrics['acc']:.4f} | "
    f"ValF1_macro {val_metrics['f1_macro']:.4f} | "
    f"ValF1_toxic {val_metrics['f1_toxic']:.4f} | "
    f"ValPrec_toxic {val_metrics['precision_toxic']:.4f} | "
    f"ValRec_toxic {val_metrics['recall_toxic']:.4f}"
    )
    try:
      val_auroc = roc_auc_score(val_metrics["y_true"], val_metrics["y_probs"])
      print(f"Val AUROC: {val_auroc:.4f}")
    except ValueError:
      print("Val AUROC: not defined (only one class present in y_true)")

    current = val_metrics["f1_macro"]
    if current > best_metric:
        best_metric = current; patience_ctr = 0
        best_state = {k: v.cpu() for k,v in model.state_dict().items()}
    else:
        patience_ctr += 1
        if patience_ctr > PATIENCE:
            print(f"Early stopping at epoch {epoch}. Best Val macro-F1: {best_metric:.4f}")
            break

if best_state is not None:
    model.load_state_dict(best_state); model.to(device)
    print("Loaded best checkpoint (Val macro-F1=%.4f)" % best_metric)

Epoch  1 | TrainLoss 2.0863 | ValAcc 0.0517 | ValF1_macro 0.0491 | ValF1_toxic 0.0983 | ValPrec_toxic 0.0517 | ValRec_toxic 1.0000
Val AUROC: 0.9248
Epoch  2 | TrainLoss 0.2454 | ValAcc 0.8606 | ValF1_macro 0.6616 | ValF1_toxic 0.4021 | ValPrec_toxic 0.2583 | ValRec_toxic 0.9070
Val AUROC: 0.9354
Epoch  3 | TrainLoss 0.2373 | ValAcc 0.8966 | ValF1_macro 0.6887 | ValF1_toxic 0.4342 | ValPrec_toxic 0.3028 | ValRec_toxic 0.7674
Val AUROC: 0.8522
Epoch  4 | TrainLoss 0.1014 | ValAcc 0.9315 | ValF1_macro 0.7535 | ValF1_toxic 0.5440 | ValPrec_toxic 0.4146 | ValRec_toxic 0.7907
Val AUROC: 0.9345
Epoch  5 | TrainLoss 0.0983 | ValAcc 0.9327 | ValF1_macro 0.7630 | ValF1_toxic 0.5625 | ValPrec_toxic 0.4235 | ValRec_toxic 0.8372
Val AUROC: 0.9607
Epoch  6 | TrainLoss 0.0569 | ValAcc 0.9483 | ValF1_macro 0.7775 | ValF1_toxic 0.5825 | ValPrec_toxic 0.5000 | ValRec_toxic 0.6977
Val AUROC: 0.9545
Epoch  7 | TrainLoss 0.0595 | ValAcc 0.9543 | ValF1_macro 0.7940 | ValF1_toxic 0.6122 | ValPrec_toxic 0.54

### *Validation and Testing*

In [ ]:
# Tune threshold on VALIDATION set (Fbeta)
val_eval  = evaluate(model, val_loader, device, report=False, threshold=None)
best_thr  = best_threshold_fbeta(val_eval["y_true"], val_eval["y_probs"], beta=FBETA)
val_tuned = evaluate(model, val_loader, device, report=True, threshold=best_thr)

print("\n=== Validation (for tuning on Fbeta) ===")
print(f"Default thr=0.5 -> Val Acc: {val_eval['acc']:.4f} | Val Macro-F1: {val_eval['f1_macro']:.4f} | Val F1-toxic: {val_eval['f1_toxic']:.4f}")
print(f"Best thr={best_thr:.3f} (beta={FBETA}) -> Val Acc: {val_tuned['acc']:.4f} | Val Macro-F1: {val_tuned['f1_macro']:.4f} | Val F1-toxic: {val_tuned['f1_toxic']:.4f}")
try:
    val_auroc = roc_auc_score(val_eval["y_true"], val_eval["y_probs"])
    print(f"Val AUROC: {val_auroc:.4f}")
except ValueError:
    print("Val AUROC: not defined (only one class present)")

# FINAL TEST (use the tuned threshold)
test_eval_default = evaluate(model, test_loader, device, report=True, threshold=None)
test_eval_tuned   = evaluate(model, test_loader, device, report=True, threshold=best_thr)

print("\n=== FINAL TEST (unseen) ===")
print(f"Default thr=0.5 -> Test Acc: {test_eval_default['acc']:.4f} | Test Macro-F1: {test_eval_default['f1_macro']:.4f} | Test F1-toxic: {test_eval_default['f1_toxic']:.4f}")
print(f"Tuned thr={best_thr:.3f} (beta={FBETA}) -> Test Acc: {test_eval_tuned['acc']:.4f} | Test Macro-F1: {test_eval_tuned['f1_macro']:.4f} | Test F1-toxic: {test_eval_tuned['f1_toxic']:.4f}")
try:
    test_auroc = roc_auc_score(test_eval_default["y_true"], test_eval_default["y_probs"])
    print(f"Test AUROC: {test_auroc:.4f}")
except ValueError:
    print("Test AUROC: not defined (only one class present)")

              precision    recall  f1-score   support

           0       0.98      0.98      0.98       789
           1       0.63      0.67      0.65        43

    accuracy                           0.96       832
   macro avg       0.81      0.83      0.82       832
weighted avg       0.96      0.96      0.96       832


=== Validation (for tuning on Fbeta) ===
Default thr=0.5 -> Val Acc: 0.9591 | Val Macro-F1: 0.8044 | Val F1-toxic: 0.6304
Best thr=0.748 (beta=2.0) -> Val Acc: 0.9627 | Val Macro-F1: 0.8160 | Val F1-toxic: 0.6517
Val AUROC: 0.7879
              precision    recall  f1-score   support

           0       0.98      0.98      0.98       790
           1       0.59      0.56      0.57        43

    accuracy                           0.96       833
   macro avg       0.78      0.77      0.77       833
weighted avg       0.96      0.96      0.96       833

              precision    recall  f1-score   support

           0       0.97      0.98      0.98       790
     

# **Model 2 (Greek-BERT)**

## **Classic**

In [ ]:
MODEL_NAME = GREEKB_CHECKPOINT

### *Model Setup*

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
config = AutoConfig.from_pretrained(
    MODEL_NAME, num_labels=2, label2id=label2id, id2label={v:k for k,v in label2id.items()}
)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config).to(device)

class CommentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=160):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        text = str(self.texts[idx]); label = int(self.labels[idx])
        enc = self.tokenizer(text, truncation=True, padding='max_length', max_length=self.max_length, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'labels': torch.tensor(label, dtype=torch.long)}


config.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/454M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: nlpaueb/bert-base-greek-uncased-v1
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	tho

### *Data Split*

In [ ]:
train_ds = CommentDataset(train_df["text"], train_df["label_id"], tokenizer, MAX_LEN)
val_ds   = CommentDataset(val_df["text"],   val_df["label_id"],   tokenizer, MAX_LEN)
test_ds  = CommentDataset(test_df["text"],  test_df["label_id"],  tokenizer, MAX_LEN)

# Sampler (train only)
if USE_SAMPLER:
    y_train = train_df["label_id"].values
    class_sample_count = np.bincount(y_train)     # e.g., [#non, #tox]
    weights = 1.0 / class_sample_count
    sample_weights = weights[y_train]
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SZ, sampler=sampler)
else:
    train_loader = DataLoader(train_ds, batch_size=BATCH_SZ, shuffle=True)

val_loader   = DataLoader(val_ds,   batch_size=BATCH_SZ)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SZ)

### *Evaluation Setup*

In [ ]:
@torch.no_grad()
def evaluate(model, dataloader, device, report=False, threshold=None):
    model.eval()
    all_probs, all_preds, all_true = [], [], []

    running_loss = 0.0
    for batch in dataloader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits  = outputs.logits
        loss    = F.cross_entropy(logits, labels)
        running_loss += loss.item()

        probs = torch.softmax(logits, dim=-1)[:, 1]
        all_probs.extend(probs.detach().cpu().numpy())

        preds = torch.argmax(logits, dim=1) if threshold is None else (probs >= threshold).long()
        all_preds.extend(preds.detach().cpu().numpy())
        all_true.extend(labels.detach().cpu().numpy())

    avg_loss = running_loss / max(1, len(dataloader))

    acc       = accuracy_score(all_true, all_preds)
    f1_macro  = f1_score(all_true, all_preds, average="macro",  zero_division=0)
    f1_toxic  = f1_score(all_true, all_preds, pos_label=1,      zero_division=0)
    prec_tox  = precision_score(all_true, all_preds, pos_label=1, zero_division=0)
    rec_tox   = recall_score(all_true, all_preds, pos_label=1,    zero_division=0)

    if report:
        print(classification_report(all_true, all_preds, digits=2))

    return {
        "loss": avg_loss,
        "acc": acc,
        "f1_macro": f1_macro,
        "f1_toxic": f1_toxic,
        "precision_toxic": prec_tox,
        "recall_toxic": rec_tox,
        "y_true": np.array(all_true),
        "y_probs": np.array(all_probs),
    }

def best_threshold_fbeta(y_true, p_toxic, beta=2.0):
    prec, rec, thr = precision_recall_curve(y_true, p_toxic)
    fbeta = (1+beta**2) * (prec*rec) / (beta**2 * prec + rec + 1e-9)
    if len(thr) == 0: return 0.5
    return float(thr[int(np.nanargmax(fbeta))])


### *Training*

In [ ]:
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
num_warmup = int(WARMUP_RATIO * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=num_warmup, num_training_steps=total_steps)

best_metric = -1.0
best_state = None
patience_ctr = 0

for epoch in range(1, EPOCHS+1):
    model.train(); running_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad(set_to_none=True)
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        out = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(out.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step(); scheduler.step()
        running_loss += loss.item()
    train_loss = running_loss / max(1, len(train_loader))

    val_metrics = evaluate(model, val_loader, device, report=False, threshold=None)
    print(
    f"Epoch {epoch:>2} | "
    f"TrainLoss {train_loss:.4f} | "
    f"ValAcc {val_metrics['acc']:.4f} | "
    f"ValF1_macro {val_metrics['f1_macro']:.4f} | "
    f"ValF1_toxic {val_metrics['f1_toxic']:.4f} | "
    f"ValPrec_toxic {val_metrics['precision_toxic']:.4f} | "
    f"ValRec_toxic {val_metrics['recall_toxic']:.4f}"
    )
    try:
      val_auroc = roc_auc_score(val_metrics["y_true"], val_metrics["y_probs"])
      print(f"Val AUROC: {val_auroc:.4f}")
    except ValueError:
      print("Val AUROC: not defined (only one class present in y_true)")

    current = val_metrics["f1_macro"]
    if current > best_metric:
        best_metric = current; patience_ctr = 0
        best_state = {k: v.cpu() for k,v in model.state_dict().items()}
    else:
        patience_ctr += 1
        if patience_ctr > PATIENCE:
            print(f"Early stopping at epoch {epoch}. Best Val macro-F1: {best_metric:.4f}")
            break

if best_state is not None:
    model.load_state_dict(best_state); model.to(device)
    print("Loaded best checkpoint (Val macro-F1=%.4f)" % best_metric)

model.safetensors:   0%|          | 0.00/454M [00:00<?, ?B/s]

Epoch  1 | TrainLoss 1.2820 | ValAcc 0.3341 | ValF1_macro 0.2967 | ValF1_toxic 0.1344 | ValPrec_toxic 0.0720 | ValRec_toxic 1.0000
Val AUROC: 0.9217
Epoch  2 | TrainLoss 0.2409 | ValAcc 0.8762 | ValF1_macro 0.6744 | ValF1_toxic 0.4181 | ValPrec_toxic 0.2761 | ValRec_toxic 0.8605
Val AUROC: 0.9412
Epoch  3 | TrainLoss 0.1412 | ValAcc 0.9363 | ValF1_macro 0.7525 | ValF1_toxic 0.5391 | ValPrec_toxic 0.4306 | ValRec_toxic 0.7209
Val AUROC: 0.9311
Epoch  4 | TrainLoss 0.0583 | ValAcc 0.9435 | ValF1_macro 0.7611 | ValF1_toxic 0.5524 | ValPrec_toxic 0.4677 | ValRec_toxic 0.6744
Val AUROC: 0.9195
Epoch  5 | TrainLoss 0.0537 | ValAcc 0.9447 | ValF1_macro 0.7641 | ValF1_toxic 0.5577 | ValPrec_toxic 0.4754 | ValRec_toxic 0.6744
Val AUROC: 0.9173
Epoch  6 | TrainLoss 0.0433 | ValAcc 0.9543 | ValF1_macro 0.7814 | ValF1_toxic 0.5870 | ValPrec_toxic 0.5510 | ValRec_toxic 0.6279
Val AUROC: 0.9181
Epoch  7 | TrainLoss 0.0288 | ValAcc 0.9471 | ValF1_macro 0.7659 | ValF1_toxic 0.5600 | ValPrec_toxic 0.49

### *Validation and Testing*

In [ ]:
# Tune threshold on VALIDATION set (Fbeta)
val_eval  = evaluate(model, val_loader, device, report=False, threshold=None)
best_thr  = best_threshold_fbeta(val_eval["y_true"], val_eval["y_probs"], beta=FBETA)
val_tuned = evaluate(model, val_loader, device, report=True, threshold=best_thr)

print("\n=== Validation (for tuning on Fbeta) ===")
print(f"Default thr=0.5 -> Val Acc: {val_eval['acc']:.4f} | Val Macro-F1: {val_eval['f1_macro']:.4f} | Val F1-toxic: {val_eval['f1_toxic']:.4f}")
print(f"Best thr={best_thr:.3f} (beta={FBETA}) -> Val Acc: {val_tuned['acc']:.4f} | Val Macro-F1: {val_tuned['f1_macro']:.4f} | Val F1-toxic: {val_tuned['f1_toxic']:.4f}")
try:
    val_auroc = roc_auc_score(val_eval["y_true"], val_eval["y_probs"])
    print(f"Val AUROC: {val_auroc:.4f}")
except ValueError:
    print("Val AUROC: not defined (only one class present)")

# FINAL TEST (use the tuned threshold)
test_eval_default = evaluate(model, test_loader, device, report=True, threshold=None)
test_eval_tuned   = evaluate(model, test_loader, device, report=True, threshold=best_thr)

print("\n=== FINAL TEST (unseen) ===")
print(f"Default thr=0.5 -> Test Acc: {test_eval_default['acc']:.4f} | Test Macro-F1: {test_eval_default['f1_macro']:.4f} | Test F1-toxic: {test_eval_default['f1_toxic']:.4f}")
print(f"Tuned thr={best_thr:.3f} (beta={FBETA}) -> Test Acc: {test_eval_tuned['acc']:.4f} | Test Macro-F1: {test_eval_tuned['f1_macro']:.4f} | Test F1-toxic: {test_eval_tuned['f1_toxic']:.4f}")
try:
    test_auroc = roc_auc_score(test_eval_default["y_true"], test_eval_default["y_probs"])
    print(f"Test AUROC: {test_auroc:.4f}")
except ValueError:
    print("Test AUROC: not defined (only one class present)")

              precision    recall  f1-score   support

           0       0.99      0.95      0.97       789
           1       0.43      0.74      0.55        43

    accuracy                           0.94       832
   macro avg       0.71      0.85      0.76       832
weighted avg       0.96      0.94      0.94       832


=== Validation (for tuning on Fbeta) ===
Default thr=0.5 -> Val Acc: 0.9543 | Val Macro-F1: 0.7814 | Val F1-toxic: 0.5870
Best thr=0.077 (beta=2.0) -> Val Acc: 0.9363 | Val Macro-F1: 0.7564 | Val F1-toxic: 0.5470
Val AUROC: 0.9181
              precision    recall  f1-score   support

           0       0.97      0.98      0.98       790
           1       0.57      0.40      0.47        43

    accuracy                           0.95       833
   macro avg       0.77      0.69      0.72       833
weighted avg       0.95      0.95      0.95       833

              precision    recall  f1-score   support

           0       0.97      0.96      0.97       790
     

## **Masked**

In [ ]:
MODEL_NAME = MASKED_GREEKB

### *Model Setup*

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
config = AutoConfig.from_pretrained(
    MODEL_NAME, num_labels=2, label2id=label2id, id2label={v:k for k,v in label2id.items()}
)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config).to(device)

class CommentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=160):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        text = str(self.texts[idx]); label = int(self.labels[idx])
        enc = self.tokenizer(text, truncation=True, padding='max_length', max_length=self.max_length, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'labels': torch.tensor(label, dtype=torch.long)}


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: /content/drive/MyDrive/Toxic language in football Research/models/Greek-BERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.pooler.dense.bias                     | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if

### *Data Split*

In [ ]:
train_ds = CommentDataset(train_df["text"], train_df["label_id"], tokenizer, MAX_LEN)
val_ds   = CommentDataset(val_df["text"],   val_df["label_id"],   tokenizer, MAX_LEN)
test_ds  = CommentDataset(test_df["text"],  test_df["label_id"],  tokenizer, MAX_LEN)

# Sampler (train only)
if USE_SAMPLER:
    y_train = train_df["label_id"].values
    class_sample_count = np.bincount(y_train)     # e.g., [#non, #tox]
    weights = 1.0 / class_sample_count
    sample_weights = weights[y_train]
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SZ, sampler=sampler)
else:
    train_loader = DataLoader(train_ds, batch_size=BATCH_SZ, shuffle=True)

val_loader   = DataLoader(val_ds,   batch_size=BATCH_SZ)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SZ)

### *Evaluation Setup*

In [ ]:
@torch.no_grad()
def evaluate(model, dataloader, device, report=False, threshold=None):
    model.eval()
    all_probs, all_preds, all_true = [], [], []

    running_loss = 0.0
    for batch in dataloader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits  = outputs.logits
        loss    = F.cross_entropy(logits, labels)
        running_loss += loss.item()

        probs = torch.softmax(logits, dim=-1)[:, 1]
        all_probs.extend(probs.detach().cpu().numpy())

        preds = torch.argmax(logits, dim=1) if threshold is None else (probs >= threshold).long()
        all_preds.extend(preds.detach().cpu().numpy())
        all_true.extend(labels.detach().cpu().numpy())

    avg_loss = running_loss / max(1, len(dataloader))

    acc       = accuracy_score(all_true, all_preds)
    f1_macro  = f1_score(all_true, all_preds, average="macro",  zero_division=0)
    f1_toxic  = f1_score(all_true, all_preds, pos_label=1,      zero_division=0)
    prec_tox  = precision_score(all_true, all_preds, pos_label=1, zero_division=0)
    rec_tox   = recall_score(all_true, all_preds, pos_label=1,    zero_division=0)

    if report:
        print(classification_report(all_true, all_preds, digits=2))

    return {
        "loss": avg_loss,
        "acc": acc,
        "f1_macro": f1_macro,
        "f1_toxic": f1_toxic,
        "precision_toxic": prec_tox,
        "recall_toxic": rec_tox,
        "y_true": np.array(all_true),
        "y_probs": np.array(all_probs),
    }

def best_threshold_fbeta(y_true, p_toxic, beta=2.0):
    prec, rec, thr = precision_recall_curve(y_true, p_toxic)
    fbeta = (1+beta**2) * (prec*rec) / (beta**2 * prec + rec + 1e-9)
    if len(thr) == 0: return 0.5
    return float(thr[int(np.nanargmax(fbeta))])


### *Training*

In [ ]:
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
num_warmup = int(WARMUP_RATIO * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=num_warmup, num_training_steps=total_steps)

best_metric = -1.0
best_state = None
patience_ctr = 0

for epoch in range(1, EPOCHS+1):
    model.train(); running_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad(set_to_none=True)
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        out = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(out.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step(); scheduler.step()
        running_loss += loss.item()
    train_loss = running_loss / max(1, len(train_loader))

    val_metrics = evaluate(model, val_loader, device, report=False, threshold=None)
    print(
    f"Epoch {epoch:>2} | "
    f"TrainLoss {train_loss:.4f} | "
    f"ValAcc {val_metrics['acc']:.4f} | "
    f"ValF1_macro {val_metrics['f1_macro']:.4f} | "
    f"ValF1_toxic {val_metrics['f1_toxic']:.4f} | "
    f"ValPrec_toxic {val_metrics['precision_toxic']:.4f} | "
    f"ValRec_toxic {val_metrics['recall_toxic']:.4f}"
    )
    try:
      val_auroc = roc_auc_score(val_metrics["y_true"], val_metrics["y_probs"])
      print(f"Val AUROC: {val_auroc:.4f}")
    except ValueError:
      print("Val AUROC: not defined (only one class present in y_true)")

    current = val_metrics["f1_macro"]
    if current > best_metric:
        best_metric = current; patience_ctr = 0
        best_state = {k: v.cpu() for k,v in model.state_dict().items()}
    else:
        patience_ctr += 1
        if patience_ctr > PATIENCE:
            print(f"Early stopping at epoch {epoch}. Best Val macro-F1: {best_metric:.4f}")
            break

if best_state is not None:
    model.load_state_dict(best_state); model.to(device)
    print("Loaded best checkpoint (Val macro-F1=%.4f)" % best_metric)

Epoch  1 | TrainLoss 1.1141 | ValAcc 0.0625 | ValF1_macro 0.0609 | ValF1_toxic 0.0993 | ValPrec_toxic 0.0522 | ValRec_toxic 1.0000
Val AUROC: 0.8634
Epoch  2 | TrainLoss 0.3478 | ValAcc 0.5805 | ValF1_macro 0.4519 | ValF1_toxic 0.1865 | ValPrec_toxic 0.1036 | ValRec_toxic 0.9302
Val AUROC: 0.8955
Epoch  3 | TrainLoss 0.2265 | ValAcc 0.7740 | ValF1_macro 0.5633 | ValF1_toxic 0.2598 | ValPrec_toxic 0.1564 | ValRec_toxic 0.7674
Val AUROC: 0.8460
Epoch  4 | TrainLoss 0.1755 | ValAcc 0.8029 | ValF1_macro 0.5567 | ValF1_toxic 0.2264 | ValPrec_toxic 0.1420 | ValRec_toxic 0.5581
Val AUROC: 0.7996
Epoch  5 | TrainLoss 0.1387 | ValAcc 0.8041 | ValF1_macro 0.5811 | ValF1_toxic 0.2756 | ValPrec_toxic 0.1703 | ValRec_toxic 0.7209
Val AUROC: 0.8386
Epoch  6 | TrainLoss 0.1319 | ValAcc 0.8269 | ValF1_macro 0.5799 | ValF1_toxic 0.2577 | ValPrec_toxic 0.1656 | ValRec_toxic 0.5814
Val AUROC: 0.8159
Epoch  7 | TrainLoss 0.1006 | ValAcc 0.8281 | ValF1_macro 0.5732 | ValF1_toxic 0.2434 | ValPrec_toxic 0.15

### *Validation and Testing*

In [ ]:
# Tune threshold on VALIDATION set (Fbeta)
val_eval  = evaluate(model, val_loader, device, report=False, threshold=None)
best_thr  = best_threshold_fbeta(val_eval["y_true"], val_eval["y_probs"], beta=FBETA)
val_tuned = evaluate(model, val_loader, device, report=True, threshold=best_thr)

print("\n=== Validation (for tuning on Fbeta) ===")
print(f"Default thr=0.5 -> Val Acc: {val_eval['acc']:.4f} | Val Macro-F1: {val_eval['f1_macro']:.4f} | Val F1-toxic: {val_eval['f1_toxic']:.4f}")
print(f"Best thr={best_thr:.3f} (beta={FBETA}) -> Val Acc: {val_tuned['acc']:.4f} | Val Macro-F1: {val_tuned['f1_macro']:.4f} | Val F1-toxic: {val_tuned['f1_toxic']:.4f}")
try:
    val_auroc = roc_auc_score(val_eval["y_true"], val_eval["y_probs"])
    print(f"Val AUROC: {val_auroc:.4f}")
except ValueError:
    print("Val AUROC: not defined (only one class present)")

# FINAL TEST (use the tuned threshold)
test_eval_default = evaluate(model, test_loader, device, report=True, threshold=None)
test_eval_tuned   = evaluate(model, test_loader, device, report=True, threshold=best_thr)

print("\n=== FINAL TEST (unseen) ===")
print(f"Default thr=0.5 -> Test Acc: {test_eval_default['acc']:.4f} | Test Macro-F1: {test_eval_default['f1_macro']:.4f} | Test F1-toxic: {test_eval_default['f1_toxic']:.4f}")
print(f"Tuned thr={best_thr:.3f} (beta={FBETA}) -> Test Acc: {test_eval_tuned['acc']:.4f} | Test Macro-F1: {test_eval_tuned['f1_macro']:.4f} | Test F1-toxic: {test_eval_tuned['f1_toxic']:.4f}")
try:
    test_auroc = roc_auc_score(test_eval_default["y_true"], test_eval_default["y_probs"])
    print(f"Test AUROC: {test_auroc:.4f}")
except ValueError:
    print("Test AUROC: not defined (only one class present)")

              precision    recall  f1-score   support

           0       0.97      0.97      0.97       789
           1       0.41      0.40      0.40        43

    accuracy                           0.94       832
   macro avg       0.69      0.68      0.69       832
weighted avg       0.94      0.94      0.94       832


=== Validation (for tuning on Fbeta) ===
Default thr=0.5 -> Val Acc: 0.8401 | Val Macro-F1: 0.5837 | Val F1-toxic: 0.2570
Best thr=0.963 (beta=2.0) -> Val Acc: 0.9399 | Val Macro-F1: 0.6866 | Val F1-toxic: 0.4048
Val AUROC: 0.7742
              precision    recall  f1-score   support

           0       0.97      0.84      0.90       790
           1       0.13      0.47      0.21        43

    accuracy                           0.82       833
   macro avg       0.55      0.65      0.55       833
weighted avg       0.92      0.82      0.86       833

              precision    recall  f1-score   support

           0       0.96      0.96      0.96       790
     

# **Model 3 (mmBERT)**

## **Classic**

In [ ]:
MODEL_NAME = MMB_CHECKPOINT

### *Model Setup*

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
config = AutoConfig.from_pretrained(
    MODEL_NAME, num_labels=2, label2id=label2id, id2label={v:k for k,v in label2id.items()}
)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config).to(device)

class CommentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=160):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        text = str(self.texts[idx]); label = int(self.labels[idx])
        enc = self.tokenizer(text, truncation=True, padding='max_length', max_length=self.max_length, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'labels': torch.tensor(label, dtype=torch.long)}


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.23G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: jhu-clsp/mmBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
decoder.weight    | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### *Dataset Split*

In [ ]:
train_ds = CommentDataset(train_df["text"], train_df["label_id"], tokenizer, MAX_LEN)
val_ds   = CommentDataset(val_df["text"],   val_df["label_id"],   tokenizer, MAX_LEN)
test_ds  = CommentDataset(test_df["text"],  test_df["label_id"],  tokenizer, MAX_LEN)

# Sampler (train only)
if USE_SAMPLER:
    y_train = train_df["label_id"].values
    class_sample_count = np.bincount(y_train)     # e.g., [#non, #tox]
    weights = 1.0 / class_sample_count
    sample_weights = weights[y_train]
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SZ, sampler=sampler)
else:
    train_loader = DataLoader(train_ds, batch_size=BATCH_SZ, shuffle=True)

val_loader   = DataLoader(val_ds,   batch_size=BATCH_SZ)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SZ)

### *Evaluation Setup*

In [ ]:
@torch.no_grad()
def evaluate(model, dataloader, device, report=False, threshold=None):
    model.eval()
    all_probs, all_preds, all_true = [], [], []

    running_loss = 0.0
    for batch in dataloader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits  = outputs.logits
        loss    = F.cross_entropy(logits, labels)
        running_loss += loss.item()

        probs = torch.softmax(logits, dim=-1)[:, 1]
        all_probs.extend(probs.detach().cpu().numpy())

        preds = torch.argmax(logits, dim=1) if threshold is None else (probs >= threshold).long()
        all_preds.extend(preds.detach().cpu().numpy())
        all_true.extend(labels.detach().cpu().numpy())

    avg_loss = running_loss / max(1, len(dataloader))

    acc       = accuracy_score(all_true, all_preds)
    f1_macro  = f1_score(all_true, all_preds, average="macro",  zero_division=0)
    f1_toxic  = f1_score(all_true, all_preds, pos_label=1,      zero_division=0)
    prec_tox  = precision_score(all_true, all_preds, pos_label=1, zero_division=0)
    rec_tox   = recall_score(all_true, all_preds, pos_label=1,    zero_division=0)

    if report:
        print(classification_report(all_true, all_preds, digits=2))

    return {
        "loss": avg_loss,
        "acc": acc,
        "f1_macro": f1_macro,
        "f1_toxic": f1_toxic,
        "precision_toxic": prec_tox,
        "recall_toxic": rec_tox,
        "y_true": np.array(all_true),
        "y_probs": np.array(all_probs),
    }

def best_threshold_fbeta(y_true, p_toxic, beta=2.0):
    prec, rec, thr = precision_recall_curve(y_true, p_toxic)
    fbeta = (1+beta**2) * (prec*rec) / (beta**2 * prec + rec + 1e-9)
    if len(thr) == 0: return 0.5
    return float(thr[int(np.nanargmax(fbeta))])


### *Training*

In [ ]:
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
num_warmup = int(WARMUP_RATIO * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=num_warmup, num_training_steps=total_steps)

best_metric = -1.0
best_state = None
patience_ctr = 0

for epoch in range(1, EPOCHS+1):
    model.train(); running_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad(set_to_none=True)
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        out = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(out.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step(); scheduler.step()
        running_loss += loss.item()
    train_loss = running_loss / max(1, len(train_loader))

    val_metrics = evaluate(model, val_loader, device, report=False, threshold=None)
    print(
    f"Epoch {epoch:>2} | "
    f"TrainLoss {train_loss:.4f} | "
    f"ValAcc {val_metrics['acc']:.4f} | "
    f"ValF1_macro {val_metrics['f1_macro']:.4f} | "
    f"ValF1_toxic {val_metrics['f1_toxic']:.4f} | "
    f"ValPrec_toxic {val_metrics['precision_toxic']:.4f} | "
    f"ValRec_toxic {val_metrics['recall_toxic']:.4f}"
    )
    try:
      val_auroc = roc_auc_score(val_metrics["y_true"], val_metrics["y_probs"])
      print(f"Val AUROC: {val_auroc:.4f}")
    except ValueError:
      print("Val AUROC: not defined (only one class present in y_true)")

    current = val_metrics["f1_macro"]
    if current > best_metric:
        best_metric = current; patience_ctr = 0
        best_state = {k: v.cpu() for k,v in model.state_dict().items()}
    else:
        patience_ctr += 1
        if patience_ctr > PATIENCE:
            print(f"Early stopping at epoch {epoch}. Best Val macro-F1: {best_metric:.4f}")
            break

if best_state is not None:
    model.load_state_dict(best_state); model.to(device)
    print("Loaded best checkpoint (Val macro-F1=%.4f)" % best_metric)

model.safetensors:   0%|          | 0.00/1.23G [00:00<?, ?B/s]

Epoch  1 | TrainLoss 0.8456 | ValAcc 0.8762 | ValF1_macro 0.6744 | ValF1_toxic 0.4181 | ValPrec_toxic 0.2761 | ValRec_toxic 0.8605
Val AUROC: 0.9540
Epoch  2 | TrainLoss 0.0926 | ValAcc 0.8978 | ValF1_macro 0.7077 | ValF1_toxic 0.4720 | ValPrec_toxic 0.3220 | ValRec_toxic 0.8837
Val AUROC: 0.9495
Epoch  3 | TrainLoss 0.0184 | ValAcc 0.9471 | ValF1_macro 0.7702 | ValF1_toxic 0.5686 | ValPrec_toxic 0.4915 | ValRec_toxic 0.6744
Val AUROC: 0.9596
Epoch  4 | TrainLoss 0.0373 | ValAcc 0.9531 | ValF1_macro 0.7345 | ValF1_toxic 0.4935 | ValPrec_toxic 0.5588 | ValRec_toxic 0.4419
Val AUROC: 0.9483
Epoch  5 | TrainLoss 0.0278 | ValAcc 0.9615 | ValF1_macro 0.7737 | ValF1_toxic 0.5676 | ValPrec_toxic 0.6774 | ValRec_toxic 0.4884
Val AUROC: 0.9527
Epoch  6 | TrainLoss 0.0001 | ValAcc 0.9627 | ValF1_macro 0.7518 | ValF1_toxic 0.5231 | ValPrec_toxic 0.7727 | ValRec_toxic 0.3953
Val AUROC: 0.9469
Epoch  7 | TrainLoss 0.0037 | ValAcc 0.9627 | ValF1_macro 0.7518 | ValF1_toxic 0.5231 | ValPrec_toxic 0.77

### *Validation and Testing*

In [ ]:
# Tune threshold on VALIDATION set (Fbeta)
val_eval  = evaluate(model, val_loader, device, report=False, threshold=None)
best_thr  = best_threshold_fbeta(val_eval["y_true"], val_eval["y_probs"], beta=FBETA)
val_tuned = evaluate(model, val_loader, device, report=True, threshold=best_thr)

print("\n=== Validation (for tuning on Fbeta) ===")
print(f"Default thr=0.5 -> Val Acc: {val_eval['acc']:.4f} | Val Macro-F1: {val_eval['f1_macro']:.4f} | Val F1-toxic: {val_eval['f1_toxic']:.4f}")
print(f"Best thr={best_thr:.3f} (beta={FBETA}) -> Val Acc: {val_tuned['acc']:.4f} | Val Macro-F1: {val_tuned['f1_macro']:.4f} | Val F1-toxic: {val_tuned['f1_toxic']:.4f}")
try:
    val_auroc = roc_auc_score(val_eval["y_true"], val_eval["y_probs"])
    print(f"Val AUROC: {val_auroc:.4f}")
except ValueError:
    print("Val AUROC: not defined (only one class present)")

# FINAL TEST (use the tuned threshold)
test_eval_default = evaluate(model, test_loader, device, report=True, threshold=None)
test_eval_tuned   = evaluate(model, test_loader, device, report=True, threshold=best_thr)

print("\n=== FINAL TEST (unseen) ===")
print(f"Default thr=0.5 -> Test Acc: {test_eval_default['acc']:.4f} | Test Macro-F1: {test_eval_default['f1_macro']:.4f} | Test F1-toxic: {test_eval_default['f1_toxic']:.4f}")
print(f"Tuned thr={best_thr:.3f} (beta={FBETA}) -> Test Acc: {test_eval_tuned['acc']:.4f} | Test Macro-F1: {test_eval_tuned['f1_macro']:.4f} | Test F1-toxic: {test_eval_tuned['f1_toxic']:.4f}")
try:
    test_auroc = roc_auc_score(test_eval_default["y_true"], test_eval_default["y_probs"])
    print(f"Test AUROC: {test_auroc:.4f}")
except ValueError:
    print("Test AUROC: not defined (only one class present)")

              precision    recall  f1-score   support

           0       0.99      0.96      0.97       789
           1       0.49      0.77      0.59        43

    accuracy                           0.95       832
   macro avg       0.74      0.86      0.78       832
weighted avg       0.96      0.95      0.95       832


=== Validation (for tuning on Fbeta) ===
Default thr=0.5 -> Val Acc: 0.9615 | Val Macro-F1: 0.7737 | Val F1-toxic: 0.5676
Best thr=0.013 (beta=2.0) -> Val Acc: 0.9459 | Val Macro-F1: 0.7828 | Val F1-toxic: 0.5946
Val AUROC: 0.9527
              precision    recall  f1-score   support

           0       0.97      0.99      0.98       790
           1       0.79      0.35      0.48        43

    accuracy                           0.96       833
   macro avg       0.88      0.67      0.73       833
weighted avg       0.96      0.96      0.95       833

              precision    recall  f1-score   support

           0       0.98      0.96      0.97       790
     

## **Masked**

In [ ]:
MODEL_NAME = MASKED_MMB

### *Model Setup*

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
config = AutoConfig.from_pretrained(
    MODEL_NAME, num_labels=2, label2id=label2id, id2label={v:k for k,v in label2id.items()}
)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config).to(device)

class CommentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=160):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        text = str(self.texts[idx]); label = int(self.labels[idx])
        enc = self.tokenizer(text, truncation=True, padding='max_length', max_length=self.max_length, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'labels': torch.tensor(label, dtype=torch.long)}


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: /content/drive/MyDrive/Toxic language in football Research/models/mmBERT
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### *Dataset Split*

In [ ]:
train_ds = CommentDataset(train_df["text"], train_df["label_id"], tokenizer, MAX_LEN)
val_ds   = CommentDataset(val_df["text"],   val_df["label_id"],   tokenizer, MAX_LEN)
test_ds  = CommentDataset(test_df["text"],  test_df["label_id"],  tokenizer, MAX_LEN)

# Sampler (train only)
if USE_SAMPLER:
    y_train = train_df["label_id"].values
    class_sample_count = np.bincount(y_train)     # e.g., [#non, #tox]
    weights = 1.0 / class_sample_count
    sample_weights = weights[y_train]
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SZ, sampler=sampler)
else:
    train_loader = DataLoader(train_ds, batch_size=BATCH_SZ, shuffle=True)

val_loader   = DataLoader(val_ds,   batch_size=BATCH_SZ)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SZ)

### *Evaluation Setup*

In [ ]:
@torch.no_grad()
def evaluate(model, dataloader, device, report=False, threshold=None):
    model.eval()
    all_probs, all_preds, all_true = [], [], []

    running_loss = 0.0
    for batch in dataloader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits  = outputs.logits
        loss    = F.cross_entropy(logits, labels)
        running_loss += loss.item()

        probs = torch.softmax(logits, dim=-1)[:, 1]
        all_probs.extend(probs.detach().cpu().numpy())

        preds = torch.argmax(logits, dim=1) if threshold is None else (probs >= threshold).long()
        all_preds.extend(preds.detach().cpu().numpy())
        all_true.extend(labels.detach().cpu().numpy())

    avg_loss = running_loss / max(1, len(dataloader))

    acc       = accuracy_score(all_true, all_preds)
    f1_macro  = f1_score(all_true, all_preds, average="macro",  zero_division=0)
    f1_toxic  = f1_score(all_true, all_preds, pos_label=1,      zero_division=0)
    prec_tox  = precision_score(all_true, all_preds, pos_label=1, zero_division=0)
    rec_tox   = recall_score(all_true, all_preds, pos_label=1,    zero_division=0)

    if report:
        print(classification_report(all_true, all_preds, digits=2))

    return {
        "loss": avg_loss,
        "acc": acc,
        "f1_macro": f1_macro,
        "f1_toxic": f1_toxic,
        "precision_toxic": prec_tox,
        "recall_toxic": rec_tox,
        "y_true": np.array(all_true),
        "y_probs": np.array(all_probs),
    }

def best_threshold_fbeta(y_true, p_toxic, beta=2.0):
    prec, rec, thr = precision_recall_curve(y_true, p_toxic)
    fbeta = (1+beta**2) * (prec*rec) / (beta**2 * prec + rec + 1e-9)
    if len(thr) == 0: return 0.5
    return float(thr[int(np.nanargmax(fbeta))])


### *Training*

In [ ]:
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
num_warmup = int(WARMUP_RATIO * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=num_warmup, num_training_steps=total_steps)

best_metric = -1.0
best_state = None
patience_ctr = 0

for epoch in range(1, EPOCHS+1):
    model.train(); running_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad(set_to_none=True)
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        out = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(out.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step(); scheduler.step()
        running_loss += loss.item()
    train_loss = running_loss / max(1, len(train_loader))

    val_metrics = evaluate(model, val_loader, device, report=False, threshold=None)
    print(
    f"Epoch {epoch:>2} | "
    f"TrainLoss {train_loss:.4f} | "
    f"ValAcc {val_metrics['acc']:.4f} | "
    f"ValF1_macro {val_metrics['f1_macro']:.4f} | "
    f"ValF1_toxic {val_metrics['f1_toxic']:.4f} | "
    f"ValPrec_toxic {val_metrics['precision_toxic']:.4f} | "
    f"ValRec_toxic {val_metrics['recall_toxic']:.4f}"
    )
    try:
      val_auroc = roc_auc_score(val_metrics["y_true"], val_metrics["y_probs"])
      print(f"Val AUROC: {val_auroc:.4f}")
    except ValueError:
      print("Val AUROC: not defined (only one class present in y_true)")

    current = val_metrics["f1_macro"]
    if current > best_metric:
        best_metric = current; patience_ctr = 0
        best_state = {k: v.cpu() for k,v in model.state_dict().items()}
    else:
        patience_ctr += 1
        if patience_ctr > PATIENCE:
            print(f"Early stopping at epoch {epoch}. Best Val macro-F1: {best_metric:.4f}")
            break

if best_state is not None:
    model.load_state_dict(best_state); model.to(device)
    print("Loaded best checkpoint (Val macro-F1=%.4f)" % best_metric)

Epoch  1 | TrainLoss 1.0590 | ValAcc 0.8954 | ValF1_macro 0.7041 | ValF1_toxic 0.4663 | ValPrec_toxic 0.3167 | ValRec_toxic 0.8837
Val AUROC: 0.9566
Epoch  2 | TrainLoss 0.0986 | ValAcc 0.9555 | ValF1_macro 0.7935 | ValF1_toxic 0.6105 | ValPrec_toxic 0.5577 | ValRec_toxic 0.6744
Val AUROC: 0.9650
Epoch  3 | TrainLoss 0.0711 | ValAcc 0.9615 | ValF1_macro 0.7994 | ValF1_toxic 0.6190 | ValPrec_toxic 0.6341 | ValRec_toxic 0.6047
Val AUROC: 0.9616
Epoch  4 | TrainLoss 0.0219 | ValAcc 0.9651 | ValF1_macro 0.7923 | ValF1_toxic 0.6027 | ValPrec_toxic 0.7333 | ValRec_toxic 0.5116
Val AUROC: 0.9635
Epoch  5 | TrainLoss 0.0037 | ValAcc 0.9700 | ValF1_macro 0.8056 | ValF1_toxic 0.6269 | ValPrec_toxic 0.8750 | ValRec_toxic 0.4884
Val AUROC: 0.9539
Epoch  6 | TrainLoss 0.0001 | ValAcc 0.9688 | ValF1_macro 0.7949 | ValF1_toxic 0.6061 | ValPrec_toxic 0.8696 | ValRec_toxic 0.4651
Val AUROC: 0.9559
Epoch  7 | TrainLoss 0.0011 | ValAcc 0.9675 | ValF1_macro 0.7901 | ValF1_toxic 0.5970 | ValPrec_toxic 0.83

### *Validation and Testing*

In [ ]:
# Tune threshold on VALIDATION set (Fbeta)
val_eval  = evaluate(model, val_loader, device, report=False, threshold=None)
best_thr  = best_threshold_fbeta(val_eval["y_true"], val_eval["y_probs"], beta=FBETA)
val_tuned = evaluate(model, val_loader, device, report=True, threshold=best_thr)

print("\n=== Validation (for tuning on Fbeta) ===")
print(f"Default thr=0.5 -> Val Acc: {val_eval['acc']:.4f} | Val Macro-F1: {val_eval['f1_macro']:.4f} | Val F1-toxic: {val_eval['f1_toxic']:.4f}")
print(f"Best thr={best_thr:.3f} (beta={FBETA}) -> Val Acc: {val_tuned['acc']:.4f} | Val Macro-F1: {val_tuned['f1_macro']:.4f} | Val F1-toxic: {val_tuned['f1_toxic']:.4f}")
try:
    val_auroc = roc_auc_score(val_eval["y_true"], val_eval["y_probs"])
    print(f"Val AUROC: {val_auroc:.4f}")
except ValueError:
    print("Val AUROC: not defined (only one class present)")

# FINAL TEST (use the tuned threshold)
test_eval_default = evaluate(model, test_loader, device, report=True, threshold=None)
test_eval_tuned   = evaluate(model, test_loader, device, report=True, threshold=best_thr)

print("\n=== FINAL TEST (unseen) ===")
print(f"Default thr=0.5 -> Test Acc: {test_eval_default['acc']:.4f} | Test Macro-F1: {test_eval_default['f1_macro']:.4f} | Test F1-toxic: {test_eval_default['f1_toxic']:.4f}")
print(f"Tuned thr={best_thr:.3f} (beta={FBETA}) -> Test Acc: {test_eval_tuned['acc']:.4f} | Test Macro-F1: {test_eval_tuned['f1_macro']:.4f} | Test F1-toxic: {test_eval_tuned['f1_toxic']:.4f}")
try:
    test_auroc = roc_auc_score(test_eval_default["y_true"], test_eval_default["y_probs"])
    print(f"Test AUROC: {test_auroc:.4f}")
except ValueError:
    print("Test AUROC: not defined (only one class present)")

              precision    recall  f1-score   support

           0       0.99      0.95      0.97       789
           1       0.46      0.77      0.57        43

    accuracy                           0.94       832
   macro avg       0.72      0.86      0.77       832
weighted avg       0.96      0.94      0.95       832


=== Validation (for tuning on Fbeta) ===
Default thr=0.5 -> Val Acc: 0.9700 | Val Macro-F1: 0.8056 | Val F1-toxic: 0.6269
Best thr=0.003 (beta=2.0) -> Val Acc: 0.9411 | Val Macro-F1: 0.7711 | Val F1-toxic: 0.5739
Val AUROC: 0.9539
              precision    recall  f1-score   support

           0       0.96      0.99      0.98       790
           1       0.69      0.26      0.37        43

    accuracy                           0.96       833
   macro avg       0.82      0.62      0.67       833
weighted avg       0.95      0.96      0.95       833

              precision    recall  f1-score   support

           0       0.97      0.96      0.97       790
     